# SPROUT - Kaggle Training Notebook
## Symptom-centric Prototypical Representation Optimization and Uncertainty-aware Tuning

**Few-Shot Plant Disease Classification using Prototypical Networks with Attention-Based Prototype Refinement**

### Setup Instructions:
1. Enable GPU in Kaggle Settings (Settings → Accelerator → GPU)
2. Add the PlantVillage dataset as input (search for 'plantvillage' in Datasets)
3. Run all cells sequentially

## Cell 1: Install Dependencies

In [ ]:
!pip install -q timm

## Cell 2: Import Libraries & Setup Environment

In [ ]:
import os
import sys
import random
import time
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## Cell 3: Configuration & Hyperparameters

In [ ]:
class Config:
    # Dataset
    DATA_DIR = None  # Auto-detected below
    
    # Model
    BACKBONE = 'resnet50'       # 'resnet50' or 'efficientnet_b0'
    EMBED_DIM = 128
    HIDDEN_DIMS = [512, 256]
    NUM_REFINEMENT_STEPS = 3
    TEMPERATURE = 10.0
    
    # Few-shot settings
    N_WAY = 5
    K_SHOT = 5
    N_QUERY = 15
    
    # Training
    NUM_EPISODES = 100
    NUM_EPOCHS = 20
    LR = 0.001
    WEIGHT_DECAY = 1e-5
    SCHEDULER_STEP = 5
    SCHEDULER_GAMMA = 0.5
    
    # Loss weights
    ALPHA = 0.5    # Refinement loss
    BETA = 0.3     # Intra-class loss
    GAMMA = 0.2    # Inter-class loss
    MARGIN = 1.0   # Margin for inter-class separation
    
    # Mixed precision
    USE_AMP = True
    
    # Reproducibility
    SEED = 42
    
    # Output
    OUTPUT_DIR = '/kaggle/working/sprout_results'
    
    # Evaluation
    NUM_TEST_EPISODES = 100
    TEST_SHOTS = [1, 3, 5, 10]

# Auto-detect dataset path
def detect_data_dir():
    possible_paths = [
        '/kaggle/input/plantvillage/plantvillage',
        '/kaggle/input/plantvillage',
        '/kaggle/input/plantvillage-dataset/PlantVillage',
        '/kaggle/input/plantvillage-dataset/plantvillage',
    ]
    for path in possible_paths:
        if os.path.exists(path):
            # Check if train/test folders exist
            if os.path.exists(os.path.join(path, 'train')) and os.path.exists(os.path.join(path, 'test')):
                return path
            # Check one level deeper
            for sub in os.listdir(path):
                sub_path = os.path.join(path, sub)
                if os.path.isdir(sub_path) and os.path.exists(os.path.join(sub_path, 'train')):
                    return sub_path
    return None

Config.DATA_DIR = detect_data_dir()

if Config.DATA_DIR is None:
    print("ERROR: Could not find PlantVillage dataset!")
    print("Please add the PlantVillage dataset as a Kaggle input.")
    print("Search for 'plantvillage' in Kaggle Datasets and add it.")
else:
    print(f"Dataset found at: {Config.DATA_DIR}")
    print(f"Train folder: {os.path.exists(os.path.join(Config.DATA_DIR, 'train'))}")
    print(f"Test folder: {os.path.exists(os.path.join(Config.DATA_DIR, 'test'))}")

# Create output directory
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {Config.OUTPUT_DIR}")

## Cell 4: Set Seeds & Device

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(Config.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## Cell 5: Dataset Class

In [ ]:
class LeafDiseaseDataset(Dataset):
    def __init__(self, root_dir, transform=None, split='train'):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.split_dir = self.root_dir / split

        self.classes = sorted([d.name for d in self.split_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        self.image_paths = []
        self.labels = []

        for class_name in self.classes:
            class_dir = self.split_dir / class_name
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
                for img_path in class_dir.glob(f'**/{ext}'):
                    self.image_paths.append(img_path)
                    self.labels.append(self.class_to_idx[class_name])

        self.labels = torch.tensor(self.labels)
        self.indices_by_class = {}
        for class_idx in range(len(self.classes)):
            self.indices_by_class[class_idx] = torch.where(self.labels == class_idx)[0].tolist()

        print(f"  {split}: {len(self.image_paths)} images across {len(self.classes)} classes")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except Exception:
            image = torch.zeros((3, 224, 224))
        return image, label

    def get_classes(self):
        return self.classes

    def get_class_counts(self):
        return {cls: len(indices) for cls, indices in zip(self.classes, [self.indices_by_class[i] for i in range(len(self.classes))])}

## Cell 6: Load Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2)
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Loading dataset...")
train_dataset = LeafDiseaseDataset(Config.DATA_DIR, transform=train_transform, split='train')
test_dataset = LeafDiseaseDataset(Config.DATA_DIR, transform=test_transform, split='test')

classes = train_dataset.get_classes()
num_classes = len(classes)
print(f"\nClasses ({num_classes}): {classes}")

# Print class distribution
print("\nClass distribution:")
for cls, count in train_dataset.get_class_counts().items():
    print(f"  {cls}: {count} train, {test_dataset.get_class_counts()[cls]} test")

## Cell 7: Model Components - Feature Extractor

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self, backbone='resnet50', pretrained=True):
        super().__init__()
        if backbone == 'resnet50':
            weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
            base = models.resnet50(weights=weights)
            self.feature_dim = base.fc.in_features
            self.backbone = nn.Sequential(*list(base.children())[:-1])
        elif backbone == 'efficientnet_b0':
            weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
            base = models.efficientnet_b0(weights=weights)
            self.feature_dim = base.classifier[1].in_features
            self.backbone = nn.Sequential(*list(base.children())[:-1])
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

    def forward(self, x):
        features = self.backbone(x)
        return features.view(features.size(0), -1)

    def get_feature_dim(self):
        return self.feature_dim

## Cell 8: Model Components - Embedding Network

In [ ]:
class EmbeddingNetwork(nn.Module):
    def __init__(self, input_dim, embed_dim=128, hidden_dims=[512, 256]):
        super().__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dims[0]))
        layers.append(nn.BatchNorm1d(hidden_dims[0]))
        layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Dropout(0.2))
        for i in range(len(hidden_dims) - 1):
            layers.append(nn.Linear(hidden_dims[i], hidden_dims[i+1]))
            layers.append(nn.BatchNorm1d(hidden_dims[i+1]))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(0.2))
        layers.append(nn.Linear(hidden_dims[-1], embed_dim))
        self.embedding_layers = nn.Sequential(*layers)

    def forward(self, x):
        embeddings = self.embedding_layers(x)
        return F.normalize(embeddings, p=2, dim=1)

## Cell 9: Model Components - Prototype Module

In [ ]:
class PrototypeModule(nn.Module):
    def __init__(self, embed_dim=128, num_refinement_steps=3):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_refinement_steps = num_refinement_steps
        self.attention = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim // 2, 1)
        )
        self.refinement_network = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim, embed_dim)
        )

    def generate_initial_prototypes(self, support_embeddings, support_labels):
        classes = torch.unique(support_labels)
        prototypes = []
        for c in classes:
            class_mask = (support_labels == c)
            class_embeddings = support_embeddings[class_mask]
            prototype = torch.mean(class_embeddings, dim=0) if len(class_embeddings) > 0 else torch.zeros(support_embeddings.size(1), device=support_embeddings.device)
            prototypes.append(prototype)
        return torch.stack(prototypes)

    def refine_prototype(self, prototype, support_embeddings, support_labels, class_idx):
        class_mask = (support_labels == class_idx)
        class_embeddings = support_embeddings[class_mask]
        if len(class_embeddings) == 0:
            return prototype
        prototype_expanded = prototype.unsqueeze(0).expand(class_embeddings.size(0), -1)
        attention_input = torch.cat([class_embeddings, prototype_expanded], dim=1)
        attention_scores = self.attention(attention_input)
        attention_weights = F.softmax(attention_scores, dim=0)
        weighted_avg = torch.sum(attention_weights * class_embeddings, dim=0)
        refinement_input = torch.cat([prototype, weighted_avg], dim=0)
        refined = self.refinement_network(refinement_input.unsqueeze(0)).squeeze(0)
        return F.normalize(refined, p=2, dim=0)

    def forward(self, support_embeddings, support_labels):
        if len(support_embeddings) == 0:
            return torch.zeros((0, self.embed_dim), device=support_embeddings.device)
        prototypes = self.generate_initial_prototypes(support_embeddings, support_labels)
        classes = torch.unique(support_labels)
        for _ in range(self.num_refinement_steps):
            refined_prototypes = []
            for i, c in enumerate(classes):
                if i < len(prototypes):
                    refined_prototypes.append(self.refine_prototype(prototypes[i], support_embeddings, support_labels, c))
            prototypes = torch.stack(refined_prototypes) if refined_prototypes else prototypes
        return prototypes

## Cell 10: SPROUT Model

In [ ]:
class SPROUT(nn.Module):
    def __init__(self, num_classes, backbone='resnet50', embed_dim=128,
                 hidden_dims=[512, 256], num_refinement_steps=3, temperature=10.0):
        super().__init__()
        self.feature_extractor = FeatureExtractor(backbone=backbone)
        feature_dim = self.feature_extractor.get_feature_dim()
        self.embedding_network = EmbeddingNetwork(input_dim=feature_dim, embed_dim=embed_dim, hidden_dims=hidden_dims)
        self.prototype_module = PrototypeModule(embed_dim=embed_dim, num_refinement_steps=num_refinement_steps)
        self.temperature = temperature

    def forward(self, query_images, support_images=None, support_labels=None):
        query_features = self.feature_extractor(query_images)
        query_embeddings = self.embedding_network(query_features)
        if support_images is not None and support_labels is not None:
            support_features = self.feature_extractor(support_images)
            support_embeddings = self.embedding_network(support_features)
            prototypes = self.prototype_module(support_embeddings, support_labels)
            logits = -self.compute_distances(query_embeddings, prototypes)
            return logits, prototypes
        return query_embeddings

    def compute_distances(self, embeddings, prototypes):
        embeddings_expanded = embeddings.unsqueeze(1)
        prototypes_expanded = prototypes.unsqueeze(0)
        distances = torch.sum((embeddings_expanded - prototypes_expanded) ** 2, dim=2)
        return distances / self.temperature

    def extract_embeddings(self, images):
        self.eval()
        with torch.no_grad():
            features = self.feature_extractor(images)
            embeddings = self.embedding_network(features)
        return embeddings

## Cell 11: SPROUT Loss Function

In [ ]:
class SPROUTLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.3, gamma=0.2, margin=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.margin = margin

    def forward(self, logits, targets, prototypes, initial_prototypes, support_embeddings, support_labels):
        proto_loss = F.cross_entropy(logits, targets)
        refine_loss = self.alpha * torch.mean(1 - F.cosine_similarity(prototypes, initial_prototypes, dim=1))
        classes = torch.unique(support_labels)
        intra_loss = 0.0
        for c in classes:
            mask = (support_labels == c)
            if torch.sum(mask) > 1:
                class_emb = support_embeddings[mask]
                centroid = torch.mean(class_emb, dim=0, keepdim=True)
                intra_loss += torch.var(torch.sum((class_emb - centroid) ** 2, dim=1))
        intra_loss = self.beta * intra_loss / len(classes) if len(classes) > 0 else torch.tensor(0.0, device=logits.device)

        if prototypes.size(0) > 1:
            distances = torch.cdist(prototypes, prototypes, p=2)
            mask_upper = torch.triu(torch.ones_like(distances), diagonal=1) == 1
            inter_loss = self.gamma * F.relu(self.margin - distances[mask_upper]).mean()
        else:
            inter_loss = torch.tensor(0.0, device=logits.device)

        total_loss = proto_loss + refine_loss + intra_loss + inter_loss
        return total_loss, {
            'proto_loss': proto_loss.item(),
            'refine_loss': refine_loss.item(),
            'intra_loss': intra_loss.item() if isinstance(intra_loss, torch.Tensor) else intra_loss,
            'inter_loss': inter_loss.item() if isinstance(inter_loss, torch.Tensor) else inter_loss,
            'total_loss': total_loss.item()
        }

## Cell 12: Episode Creation

In [ ]:
def create_episode(dataset, n_way, k_shot, n_query):
    indices_by_class = dataset.indices_by_class
    available_classes = [c for c in indices_by_class if len(indices_by_class[c]) >= k_shot + n_query]

    if len(available_classes) < n_way:
        available_classes = [c for c in indices_by_class if len(indices_by_class[c]) >= 2]
        n_way = min(n_way, len(available_classes))

    selected_classes = random.sample(available_classes, n_way)

    support_images, support_labels = [], []
    query_images, query_labels = [], []

    for new_label, class_idx in enumerate(selected_classes):
        indices = indices_by_class[class_idx]
        random.shuffle(indices)
        support_indices = indices[:k_shot]
        query_indices = indices[k_shot:k_shot + n_query]

        for idx in support_indices:
            img, _ = dataset[idx]
            support_images.append(img)
            support_labels.append(new_label)

        for idx in query_indices:
            img, _ = dataset[idx]
            query_images.append(img)
            query_labels.append(new_label)

    support_images = torch.stack(support_images).to(device)
    support_labels = torch.tensor(support_labels).to(device)
    query_images = torch.stack(query_images).to(device)
    query_labels = torch.tensor(query_labels).to(device)

    return support_images, support_labels, query_images, query_labels

print("Episode creation function defined!")

## Cell 13: Create Model

In [ ]:
model = SPROUT(
    num_classes=num_classes,
    backbone=Config.BACKBONE,
    embed_dim=Config.EMBED_DIM,
    hidden_dims=Config.HIDDEN_DIMS,
    num_refinement_steps=Config.NUM_REFINEMENT_STEPS,
    temperature=Config.TEMPERATURE
).to(device)

criterion = SPROUTLoss(alpha=Config.ALPHA, beta=Config.BETA, gamma=Config.GAMMA, margin=Config.MARGIN)
optimizer = optim.Adam(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=Config.SCHEDULER_STEP, gamma=Config.SCHEDULER_GAMMA)

# Mixed precision scaler
scaler = GradScaler(enabled=Config.USE_AMP and torch.cuda.is_available())

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {Config.BACKBONE}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Embedding dim: {Config.EMBED_DIM}")
print(f"Refinement steps: {Config.NUM_REFINEMENT_STEPS}")
print(f"Mixed precision: {Config.USE_AMP and torch.cuda.is_available()}")

## Cell 14: Training Function

In [ ]:
def train_one_epoch(model, criterion, optimizer, scaler, dataset, config):
    model.train()
    episode_accuracies = []
    episode_losses = []
    loss_components_sum = {'proto_loss': 0, 'refine_loss': 0, 'intra_loss': 0, 'inter_loss': 0, 'total_loss': 0}

    pbar = tqdm(range(config.NUM_EPISODES), desc="  Episodes", leave=False)
    for _ in pbar:
        optimizer.zero_grad()

        support_images, support_labels, query_images, query_labels = create_episode(
            dataset, config.N_WAY, config.K_SHOT, config.N_QUERY
        )

        with autocast(enabled=config.USE_AMP and torch.cuda.is_available()):
            logits, prototypes = model(query_images, support_images, support_labels)

            with torch.no_grad():
                support_features = model.feature_extractor(support_images)
                support_embeddings = model.embedding_network(support_features)
                initial_prototypes = model.prototype_module.generate_initial_prototypes(support_embeddings, support_labels)

            loss, loss_components = criterion(logits, query_labels, prototypes, initial_prototypes, support_embeddings, support_labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        _, predicted = torch.max(logits.data, 1)
        accuracy = (predicted == query_labels).float().mean().item()

        episode_accuracies.append(accuracy)
        episode_losses.append(loss_components['total_loss'])
        for k in loss_components_sum:
            loss_components_sum[k] += loss_components[k]

        pbar.set_postfix(acc=f"{accuracy:.3f}", loss=f"{loss_components['total_loss']:.4f}")

    n = config.NUM_EPISODES
    avg_loss_components = {k: v / n for k, v in loss_components_sum.items()}
    return np.mean(episode_accuracies), np.mean(episode_losses), avg_loss_components

## Cell 15: Train the Model

In [ ]:
print("Starting SPROUT Training...")
print("=" * 60)
print(f"Config: {Config.N_WAY}-way {Config.K_SHOT}-shot, {Config.NUM_EPISODES} episodes/epoch, {Config.NUM_EPOCHS} epochs")
print(f"Backbone: {Config.BACKBONE}, Embed dim: {Config.EMBED_DIM}, LR: {Config.LR}")
print("=" * 60)

train_accuracies = []
train_losses = []
best_acc = 0.0
start_time = time.time()

for epoch in range(Config.NUM_EPOCHS):
    epoch_start = time.time()
    print(f"\nEpoch {epoch+1}/{Config.NUM_EPOCHS}")
    print("-" * 40)

    epoch_acc, epoch_loss, loss_comp = train_one_epoch(model, criterion, optimizer, scaler, train_dataset, Config)

    scheduler.step()
    epoch_time = time.time() - epoch_start

    train_accuracies.append(epoch_acc)
    train_losses.append(epoch_loss)

    print(f"  Accuracy: {epoch_acc:.4f} | Loss: {epoch_loss:.4f} | Time: {epoch_time:.1f}s")
    print(f"  Loss components: proto={loss_comp['proto_loss']:.4f}, refine={loss_comp['refine_loss']:.4f}, "
          f"intra={loss_comp['intra_loss']:.4f}, inter={loss_comp['inter_loss']:.4f}")
    print(f"  LR: {scheduler.get_last_lr()[0]:.6f}")

    if torch.cuda.is_available():
        print(f"  GPU Memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated")

    # Save checkpoint
    if (epoch + 1) % 5 == 0 or epoch == Config.NUM_EPOCHS - 1:
        ckpt_path = os.path.join(Config.OUTPUT_DIR, f'sprout_epoch_{epoch+1}.pth')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'accuracy': epoch_acc,
            'config': {k: v for k, v in vars(Config).items() if not k.startswith('_')}
        }, ckpt_path)
        print(f"  Checkpoint saved: {ckpt_path}")

    if epoch_acc > best_acc:
        best_acc = epoch_acc
        best_path = os.path.join(Config.OUTPUT_DIR, 'sprout_best.pth')
        torch.save(model.state_dict(), best_path)
        print(f"  New best model saved!")

total_time = time.time() - start_time
print("\n" + "=" * 60)
print(f"Training Complete! Total time: {total_time/60:.1f} minutes")
print(f"Best training accuracy: {best_acc:.4f}")

# Save final model
final_path = os.path.join(Config.OUTPUT_DIR, 'sprout_final.pth')
torch.save(model.state_dict(), final_path)
print(f"Final model saved: {final_path}")

## Cell 16: Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, Config.NUM_EPOCHS + 1)

ax1.plot(epochs, train_accuracies, 'b-o', linewidth=2, markersize=6, label='Train Accuracy')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Training Accuracy', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1.05])

ax2.plot(epochs, train_losses, 'r-o', linewidth=2, markersize=6, label='Train Loss')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Training Loss', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle(f'SPROUT Training ({Config.BACKBONE}, {Config.N_WAY}-way {Config.K_SHOT}-shot)', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(Config.OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {Config.OUTPUT_DIR}/training_curves.png")

## Cell 17: Evaluation Functions

In [ ]:
def evaluate_few_shot(model, dataset, n_way, k_shot, n_query, num_episodes, dataset_name="Test"):
    model.eval()
    all_preds = []
    all_labels = []
    episode_accs = []

    for _ in tqdm(range(num_episodes), desc=f"  Evaluating ({dataset_name})", leave=False):
        support_images, support_labels, query_images, query_labels = create_episode(
            dataset, n_way, k_shot, n_query
        )

        with torch.no_grad():
            logits, _ = model(query_images, support_images, support_labels)

        _, predicted = torch.max(logits.data, 1)
        accuracy = (predicted == query_labels).float().mean().item()
        episode_accs.append(accuracy)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(query_labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        'precision_macro': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall_macro': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'episode_mean_acc': np.mean(episode_accs),
        'episode_std_acc': np.std(episode_accs)
    }

    return metrics, all_preds, all_labels, episode_accs

## Cell 18: Evaluate on Test Set

In [ ]:
# Load best model
best_path = os.path.join(Config.OUTPUT_DIR, 'sprout_best.pth')
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))
    print(f"Loaded best model from {best_path}")
else:
    print("Using current model (best checkpoint not found)")

model.eval()

print(f"\nEvaluating {Config.N_WAY}-way {Config.K_SHOT}-shot on test set...")
print("=" * 60)

metrics, preds, labels, episode_accs = evaluate_few_shot(
    model, test_dataset, Config.N_WAY, Config.K_SHOT, Config.N_QUERY,
    Config.NUM_TEST_EPISODES, "Test"
)

print(f"\nResults ({Config.N_WAY}-way {Config.K_SHOT}-shot, {Config.NUM_TEST_EPISODES} episodes):")
print(f"  Episode Accuracy: {metrics['episode_mean_acc']:.4f} (+/- {metrics['episode_std_acc']:.4f})")
print(f"  Overall Accuracy: {metrics['accuracy']:.4f}")
print(f"  F1 (macro):       {metrics['f1_macro']:.4f}")
print(f"  F1 (weighted):    {metrics['f1_weighted']:.4f}")
print(f"  Precision:        {metrics['precision_macro']:.4f}")
print(f"  Recall:           {metrics['recall_macro']:.4f}")

## Cell 19: Multi-Shot Evaluation

In [ ]:
print("Multi-shot evaluation on test set...")
print("=" * 60)

shot_results = {}
for shot in Config.TEST_SHOTS:
    print(f"\n  Evaluating {shot}-shot...")
    metrics, _, _, _ = evaluate_few_shot(
        model, test_dataset, Config.N_WAY, shot, Config.N_QUERY,
        Config.NUM_TEST_EPISODES, f"{shot}-shot Test"
    )
    shot_results[shot] = metrics['episode_mean_acc']
    print(f"    Accuracy: {metrics['episode_mean_acc']:.4f} (+/- {metrics['episode_std_acc']:.4f})")
    print(f"    F1 (macro): {metrics['f1_macro']:.4f}")

# Plot shot comparison
plt.figure(figsize=(10, 6))
shots = list(shot_results.keys())
accs = list(shot_results.values())
plt.plot(shots, accs, 'bo-', linewidth=2, markersize=10)
for s, a in zip(shots, accs):
    plt.text(s, a + 0.01, f'{a:.3f}', ha='center', fontsize=11)
plt.xlabel('Number of Shots (K)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title(f'SPROUT Accuracy vs. Number of Shots ({Config.N_WAY}-way)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(shots)
plt.ylim([0, 1.05])
plt.tight_layout()
plt.savefig(os.path.join(Config.OUTPUT_DIR, 'shot_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Cell 20: Confusion Matrix

In [ ]:
cm = confusion_matrix(labels, preds)
class_names = [c.split('___')[-1].replace('_', ' ') for c in test_dataset.get_classes()]
if len(class_names) > 8:
    class_names = [c[:15] + '...' if len(c) > 15 else c for c in class_names]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix ({Config.N_WAY}-way {Config.K_SHOT}-shot)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(Config.OUTPUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## Cell 21: Classification Report

In [ ]:
print("Classification Report:")
print("=" * 60)
print(classification_report(labels, preds, target_names=test_dataset.get_classes(), zero_division=0))

## Cell 22: t-SNE Visualization of Embeddings

In [ ]:
print("Generating t-SNE visualization...")
model.eval()

# Collect embeddings from test set
all_embeddings = []
all_emb_labels = []
max_samples = 500
sample_count = 0

for class_idx in range(num_classes):
    indices = test_dataset.indices_by_class[class_idx]
    sample_indices = random.sample(indices, min(50, len(indices)))
    for idx in sample_indices:
        img, label = test_dataset[idx]
        img = img.unsqueeze(0).to(device)
        with torch.no_grad():
            emb = model.extract_embeddings(img)
        all_embeddings.append(emb.cpu().numpy().flatten())
        all_emb_labels.append(label.item())
        sample_count += 1
        if sample_count >= max_samples:
            break
    if sample_count >= max_samples:
        break

all_embeddings = np.array(all_embeddings)
all_emb_labels = np.array(all_emb_labels)

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_embeddings)-1))
embeddings_2d = tsne.fit_transform(all_embeddings)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
unique_labels = np.unique(all_emb_labels)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

for i, label in enumerate(unique_labels):
    mask = all_emb_labels == label
    class_name = test_dataset.get_classes()[label]
    if len(class_name) > 20:
        class_name = class_name[:17] + '...'
    ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
               c=[colors[i]], label=class_name, alpha=0.7, s=50)

ax.set_title('t-SNE Visualization of SPROUT Embeddings', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(Config.OUTPUT_DIR, 'tsne_embeddings.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"t-SNE plot saved to {Config.OUTPUT_DIR}/tsne_embeddings.png")

## Cell 23: Distance Heatmap Visualization

In [ ]:
print("Generating distance heatmap...")
model.eval()

# Create one episode for visualization
support_images, support_labels, query_images, query_labels = create_episode(
    test_dataset, Config.N_WAY, Config.K_SHOT, 10
)

with torch.no_grad():
    logits, prototypes = model(query_images, support_images, support_labels)
    query_emb = model.extract_embeddings(query_images)

# Compute distances
prototypes_np = prototypes.cpu().numpy()
query_emb_np = query_emb.cpu().numpy()
distances = np.zeros((len(query_emb_np), len(prototypes_np)))
for i, q in enumerate(query_emb_np):
    for j, p in enumerate(prototypes_np):
        distances[i, j] = np.sum((q - p) ** 2)

# Sort by label
sort_idx = np.argsort(query_labels.cpu().numpy())
sorted_distances = distances[sort_idx]
sorted_labels = query_labels.cpu().numpy()[sort_idx]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sorted_distances, cmap='viridis_r', ax=ax)
ax.set_xlabel('Prototype Index', fontsize=12)
ax.set_ylabel('Query Sample Index', fontsize=12)
ax.set_title('Distance Heatmap: Query Samples vs. Prototypes', fontsize=14)

# Add class separators
prev_label = sorted_labels[0]
for i, label in enumerate(sorted_labels[1:], 1):
    if label != prev_label:
        ax.axhline(y=i, color='red', linestyle='-', linewidth=1.5)
        prev_label = label

plt.tight_layout()
plt.savefig(os.path.join(Config.OUTPUT_DIR, 'distance_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## Cell 24: Save Final Results

In [ ]:
results = {
    'config': {
        'backbone': Config.BACKBONE,
        'embed_dim': Config.EMBED_DIM,
        'n_way': Config.N_WAY,
        'k_shot': Config.K_SHOT,
        'n_query': Config.N_QUERY,
        'num_episodes': Config.NUM_EPISODES,
        'num_epochs': Config.NUM_EPOCHS,
        'lr': Config.LR,
        'refinement_steps': Config.NUM_REFINEMENT_STEPS,
        'temperature': Config.TEMPERATURE,
    },
    'training': {
        'final_accuracy': float(train_accuracies[-1]),
        'best_accuracy': float(max(train_accuracies)),
        'final_loss': float(train_losses[-1]),
        'total_time_minutes': total_time / 60,
    },
    'test_results': {f'{Config.N_WAY}_way_{Config.K_SHOT}_shot': metrics},
    'shot_comparison': {str(k): float(v) for k, v in shot_results.items()},
    'num_classes': num_classes,
    'classes': test_dataset.get_classes(),
    'total_params': total_params,
}

results_path = os.path.join(Config.OUTPUT_DIR, 'results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved!")
print(f"\nFiles in {Config.OUTPUT_DIR}:")
for f in sorted(os.listdir(Config.OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(Config.OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

print("\n" + "=" * 60)
print("SPROUT Training Complete!")
print(f"Best training accuracy: {max(train_accuracies):.4f}")
print(f"Test accuracy ({Config.K_SHOT}-shot): {metrics['episode_mean_acc']:.4f}")
print("=" * 60)

## Cell 25: GPU Memory Cleanup

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory cleared.")
    print(f"Final GPU Memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated")

print("\nAll done! Download the outputs from /kaggle/working/sprout_results/")